In [0]:
%pip install --upgrade typing_extensions supabase
dbutils.library.restartPython()

In [0]:
import typing_extensions
from supabase import create_client

print("Importação concluída com sucesso!")

In [0]:
# config
import pandas as pd
import numpy as np


from pyspark.sql.functions import (
    col, count, sum as sql_sum, avg,
    min as sql_min, max as sql_max,
    year, month, quarter, dayofweek,
    datediff, when, desc, asc, round
)

from pyspark.sql.window import Window

from supabase import create_client, Client




In [0]:
pip install supabase python-dotenv

In [0]:
#carregando dados
df = spark.table("fiap.silver.SINAN_DENGUE_SP")

print(f"\n Dados carregados:")
print(f"total de registros: {df.count():,.0f}")
print(f"período: {df.select(sql_min('DATA_NOTIFICACAO')).collect()[0][0]} a {df.select(sql_max('DATA_NOTIFICACAO')).collect()[0][0]}")
print(f"total de colunas: {len(df.columns)}")

#esquema
print(f"\n colunas e dados:")
df.printSchema()

In [0]:
%pip install sqlalchemy psycopg2-binary

In [0]:
from decimal import Decimal
import pandas as pd
import numpy as np
import math
import json

def enviar_para_supabase(
    tabela_spark,
    nome_tabela_supabase,
    truncate=True
):

    try:

        df_pandas = tabela_spark.toPandas()

        # converter datetime/date para string
        for col in df_pandas.columns:

            if pd.api.types.is_datetime64_any_dtype(df_pandas[col]):
                df_pandas[col] = df_pandas[col].astype(str)

            df_pandas[col] = df_pandas[col].apply(
                lambda x: str(x)
                if hasattr(x, "isoformat")
                else x
            )

        # decimal -> float
        for col in df_pandas.columns:
            df_pandas[col] = df_pandas[col].apply(
                lambda x: float(x)
                if isinstance(x, Decimal)
                else x
            )

        # nomes das colunas em lowercase
        df_pandas.columns = [
            col.lower()
            for col in df_pandas.columns
        ]

        # dataframe -> lista de dicts
        dados = df_pandas.to_dict("records")

        # limpeza final de qualquer valor inválido
        for linha in dados:

            for chave, valor in list(linha.items()):

                if isinstance(valor, np.integer):
                    linha[chave] = int(valor)

                elif isinstance(valor, np.floating):

                    if np.isnan(valor) or np.isinf(valor):
                        linha[chave] = None
                    else:
                        linha[chave] = float(valor)

                elif isinstance(valor, float):

                    if math.isnan(valor) or math.isinf(valor):
                        linha[chave] = None

        # validação JSON
        json.dumps(dados, allow_nan=False)

        print(f"\nenviando {nome_tabela_supabase}")
        print(f"total {len(dados):,.0f} registros")

        if truncate:

            try:

                primeira_coluna = df_pandas.columns[0]

                supabase.table(nome_tabela_supabase) \
                    .delete() \
                    .not_.is_(primeira_coluna, "null") \
                    .execute()

                print(f"tabela {nome_tabela_supabase} limpa")

            except Exception as e:
                print(f"não foi possivel limpar: {e}")

        chunk_size = 1000
        total_inserido = 0

        for i in range(0, len(dados), chunk_size):

            chunk = dados[i:i + chunk_size]

            supabase.table(nome_tabela_supabase) \
                .insert(chunk) \
                .execute()

            total_inserido += len(chunk)

            print(
                f"inseridos {len(chunk):,.0f} registros "
                f"({total_inserido:,.0f}/{len(dados):,.0f})"
            )

        print(f"{nome_tabela_supabase} enviado com sucesso\n")

        return True

    except Exception as e:

        print(
            f"erro ao enviar {nome_tabela_supabase}: {e}\n"
        )

        return False

##Gold

##visualizações

In [0]:
%sql
CREATE OR REPLACE TABLE fiap.gold.SINAN_DENGUE_MUNICIPIOS_MENSAL AS
SELECT
    CAST(DATE_TRUNC('month', s.DATA_NOTIFICACAO) AS DATE) AS mes_ano,
    YEAR(s.DATA_NOTIFICACAO) AS ano,
    MONTH(s.DATA_NOTIFICACAO) AS mes,

    s.COD_IBGE_MUNICIPIO,
    i.NOME_MUNICIPIO,
    i.NOME_MICRORREGIAO,
    i.NOME_MESORREGIAO,

    COUNT(*) AS casos,
    SUM(s.FLAG_HOSPITALIZACAO) AS hospitalizacoes,
    SUM(s.FLAG_OBITO_DENGUE) AS obitos,

    ROUND(
        SUM(s.FLAG_HOSPITALIZACAO) * 100.0 / COUNT(*),
        2
    ) AS taxa_hospitalizacao_pct,

    ROUND(
        SUM(s.FLAG_OBITO_DENGUE) * 100.0 / COUNT(*),
        4
    ) AS taxa_mortalidade_pct

FROM fiap.silver.SINAN_DENGUE_SP s

INNER JOIN fiap.silver.IBGE_SP i
ON CAST(s.COD_IBGE_MUNICIPIO AS BIGINT) =
   CAST(i.COD_SUS AS BIGINT)

GROUP BY
    CAST(DATE_TRUNC('month', s.DATA_NOTIFICACAO) AS DATE),
    YEAR(s.DATA_NOTIFICACAO),
    MONTH(s.DATA_NOTIFICACAO),
    s.COD_IBGE_MUNICIPIO,
    i.NOME_MUNICIPIO,
    i.NOME_MICRORREGIAO,
    i.NOME_MESORREGIAO

## CARD 01

In [0]:
%sql
CREATE OR REPLACE TABLE fiap.silver.SINAN_DENGUE_MUNICIPIOS_TOTAL_CASOS AS

WITH referencia AS (

    SELECT MAX(mes_ano) AS dt_ref
    FROM fiap.gold.SINAN_DENGUE_MUNICIPIOS_MENSAL

),

periodos AS (

    SELECT
        'Trimestre' AS periodo,
        ADD_MONTHS(dt_ref,-2) AS dt_inicio,
        dt_ref AS dt_fim,
        ADD_MONTHS(dt_ref,-5) AS dt_inicio_anterior,
        ADD_MONTHS(dt_ref,-3) AS dt_fim_anterior,
        TRUE AS possui_comparacao
    FROM referencia

    UNION ALL

    SELECT
        'Semestre',
        ADD_MONTHS(dt_ref,-5),
        dt_ref,
        ADD_MONTHS(dt_ref,-11),
        ADD_MONTHS(dt_ref,-6),
        TRUE
    FROM referencia

    UNION ALL

    SELECT
        '12 Meses',
        ADD_MONTHS(dt_ref,-11),
        dt_ref,
        ADD_MONTHS(dt_ref,-23),
        ADD_MONTHS(dt_ref,-12),
        TRUE
    FROM referencia

    UNION ALL

    SELECT
        '3 Anos',
        ADD_MONTHS(dt_ref,-35),
        dt_ref,
        NULL,
        NULL,
        FALSE
    FROM referencia

    UNION ALL

    SELECT
        '5 Anos',
        ADD_MONTHS(dt_ref,-59),
        dt_ref,
        NULL,
        NULL,
        FALSE
    FROM referencia

),

atual AS (

    SELECT

        'Dengue' AS id_agravo,

        p.periodo,

        m.cod_ibge_municipio,

        m.nome_municipio,

        p.dt_inicio,
        p.dt_fim,

        SUM(m.casos) AS casos_atual

    FROM fiap.gold.SINAN_DENGUE_MUNICIPIOS_MENSAL m

    INNER JOIN periodos p
        ON m.mes_ano BETWEEN p.dt_inicio
                         AND p.dt_fim

    GROUP BY

        p.periodo,
        m.cod_ibge_municipio,
        m.nome_municipio,
        p.dt_inicio,
        p.dt_fim

),

anterior AS (

    SELECT

        p.periodo,

        m.cod_ibge_municipio,

        SUM(m.casos) AS casos_anterior

    FROM fiap.gold.SINAN_DENGUE_MUNICIPIOS_MENSAL m

    INNER JOIN periodos p
        ON p.possui_comparacao = TRUE
       AND m.mes_ano BETWEEN p.dt_inicio_anterior
                        AND p.dt_fim_anterior

    GROUP BY

        p.periodo,
        m.cod_ibge_municipio

)

SELECT

    a.id_agravo,

    a.periodo,

    a.cod_ibge_municipio,

    a.nome_municipio,

    a.dt_inicio AS periodo_inicio,

    a.dt_fim AS periodo_fim,

    a.casos_atual,

    CASE
        WHEN a.periodo IN ('3 Anos','5 Anos')
        THEN NULL
        ELSE COALESCE(b.casos_anterior,0)
    END AS casos_anterior,

    CASE

        WHEN a.periodo IN ('3 Anos','5 Anos')
        THEN NULL

        WHEN COALESCE(b.casos_anterior,0) = 0
        THEN NULL

        ELSE ROUND(
            (
                (a.casos_atual - b.casos_anterior)
                * 100.0
            ) / b.casos_anterior,
            2
        )

    END AS variacao_pct,

    CASE
        WHEN a.periodo IN ('3 Anos','5 Anos')
        THEN FALSE

        WHEN COALESCE(b.casos_anterior,0) = 0
        THEN FALSE

        ELSE TRUE
    END AS possui_base_comparacao,

    CURRENT_DATE() AS data_referencia

FROM atual a

LEFT JOIN anterior b

    ON a.periodo = b.periodo
   AND a.cod_ibge_municipio = b.cod_ibge_municipio

ORDER BY

    periodo,
    casos_atual DESC
;

In [0]:
# Lendo a tabela do Databricks
df_total_casos = spark.table("fiap.silver.SINAN_DENGUE_MUNICIPIOS_TOTAL_CASOS")

# Enviando para o Supabase
enviar_para_supabase(df_total_casos, "sinan_dengue_municipios_total_casos")

## CARD 02

In [0]:
%sql

CREATE OR REPLACE TABLE fiap.silver.SINAN_DENGUE_MUNICIPIOS_TAXA_HOSPITALIZACAO AS

WITH referencia AS (

    SELECT MAX(mes_ano) AS dt_ref
    FROM fiap.gold.SINAN_DENGUE_MUNICIPIOS_MENSAL

),

municipios_sp AS (

    SELECT DISTINCT
        cod_ibge_municipio,
        nome_municipio
    FROM fiap.gold.SINAN_DENGUE_MUNICIPIOS_MENSAL

),

periodos AS (

    SELECT
        'Trimestre' AS periodo,
        2 AS meses_atual,
        5 AS meses_anterior_inicio,
        3 AS meses_anterior_fim,
        TRUE AS possui_comparacao

    UNION ALL

    SELECT
        'Semestre',
        5,
        11,
        6,
        TRUE

    UNION ALL

    SELECT
        '12 Meses',
        11,
        23,
        12,
        TRUE

    UNION ALL

    SELECT
        '3 Anos',
        35,
        NULL,
        NULL,
        FALSE

    UNION ALL

    SELECT
        '5 Anos',
        59,
        NULL,
        NULL,
        FALSE

),

atual AS (

    SELECT

        p.periodo,

        m.cod_ibge_municipio,
        m.nome_municipio,

        ADD_MONTHS(r.dt_ref,-p.meses_atual) AS periodo_inicio,
        r.dt_ref AS periodo_fim,

        SUM(m.casos) AS casos_atual,
        SUM(m.hospitalizacoes) AS hospitalizacoes_atual,

        ROUND(
            SUM(m.hospitalizacoes) * 100.0 /
            NULLIF(SUM(m.casos),0),
            2
        ) AS taxa_hosp_atual,

        CASE
            WHEN SUM(m.casos) >= 20
            THEN TRUE
            ELSE FALSE
        END AS possui_amostragem_suficiente

    FROM fiap.gold.SINAN_DENGUE_MUNICIPIOS_MENSAL m

    CROSS JOIN referencia r
    CROSS JOIN periodos p

    WHERE m.mes_ano BETWEEN
          ADD_MONTHS(r.dt_ref,-p.meses_atual)
          AND r.dt_ref

    GROUP BY
        p.periodo,
        m.cod_ibge_municipio,
        m.nome_municipio,
        r.dt_ref,
        p.meses_atual

),

anterior AS (

    SELECT

        p.periodo,

        m.cod_ibge_municipio,

        SUM(m.casos) AS casos_anterior,
        SUM(m.hospitalizacoes) AS hospitalizacoes_anterior,

        ROUND(
            SUM(m.hospitalizacoes) * 100.0 /
            NULLIF(SUM(m.casos),0),
            2
        ) AS taxa_hosp_anterior

    FROM fiap.gold.SINAN_DENGUE_MUNICIPIOS_MENSAL m

    CROSS JOIN referencia r
    CROSS JOIN periodos p

    WHERE p.possui_comparacao = TRUE

      AND m.mes_ano BETWEEN
          ADD_MONTHS(r.dt_ref,-p.meses_anterior_inicio)
          AND ADD_MONTHS(r.dt_ref,-p.meses_anterior_fim)

    GROUP BY
        p.periodo,
        m.cod_ibge_municipio

)

SELECT

    'Dengue' AS id_agravo,

    a.periodo,

    a.cod_ibge_municipio,
    a.nome_municipio,

    a.periodo_inicio,
    a.periodo_fim,

    a.casos_atual,
    a.hospitalizacoes_atual,
    a.taxa_hosp_atual,

    CASE
        WHEN p.possui_comparacao
        THEN b.casos_anterior
        ELSE NULL
    END AS casos_anterior,

    CASE
        WHEN p.possui_comparacao
        THEN b.hospitalizacoes_anterior
        ELSE NULL
    END AS hospitalizacoes_anterior,

    CASE
        WHEN p.possui_comparacao
        THEN b.taxa_hosp_anterior
        ELSE NULL
    END AS taxa_hosp_anterior,

    CASE
        WHEN p.possui_comparacao
             AND b.taxa_hosp_anterior IS NOT NULL
        THEN ROUND(
            a.taxa_hosp_atual -
            b.taxa_hosp_anterior,
            2
        )
        ELSE NULL
    END AS variacao_pp,

    p.possui_comparacao
        AS possui_base_comparacao,

    a.possui_amostragem_suficiente,

    CASE
        WHEN a.possui_amostragem_suficiente = FALSE
        THEN 'Baixa amostragem'
        ELSE 'OK'
    END AS observacao,

    CURRENT_TIMESTAMP() AS data_referencia

FROM atual a

INNER JOIN periodos p
    ON a.periodo = p.periodo

LEFT JOIN anterior b
    ON a.periodo = b.periodo
   AND a.cod_ibge_municipio = b.cod_ibge_municipio

ORDER BY
    periodo,
    nome_municipio;



In [0]:
# Lendo a tabela do Databricks
df_taxa_hosp = spark.table("fiap.silver.SINAN_DENGUE_MUNICIPIOS_TAXA_HOSPITALIZACAO")

# Enviando para o Supabase (garantindo que o tratamento de nulos/NaN esteja na sua função enviar_para_supabase)
enviar_para_supabase(df_taxa_hosp, "sinan_dengue_municipios_taxa_hospitalizacao")

## CARD 03

In [0]:
%sql

CREATE OR REPLACE TABLE fiap.silver.SINAN_DENGUE_MUNICIPIOS_TAXA_OBITO AS

WITH referencia AS (

    SELECT MAX(mes_ano) AS dt_ref
    FROM fiap.gold.SINAN_DENGUE_MUNICIPIOS_MENSAL

),

periodos AS (

    SELECT
        'Trimestre' AS periodo,
        2 AS meses_atual,
        5 AS meses_anterior_inicio,
        3 AS meses_anterior_fim,
        TRUE AS possui_comparacao

    UNION ALL

    SELECT
        'Semestre',
        5,
        11,
        6,
        TRUE

    UNION ALL

    SELECT
        '12 Meses',
        11,
        23,
        12,
        TRUE

    UNION ALL

    SELECT
        '3 Anos',
        35,
        NULL,
        NULL,
        FALSE

    UNION ALL

    SELECT
        '5 Anos',
        59,
        NULL,
        NULL,
        FALSE

),

atual AS (

    SELECT

        p.periodo,

        m.cod_ibge_municipio,
        m.nome_municipio,

        ADD_MONTHS(r.dt_ref,-p.meses_atual) AS periodo_inicio,
        r.dt_ref AS periodo_fim,

        SUM(m.casos) AS casos_atual,
        SUM(m.obitos) AS obitos_atual,

        ROUND(
            SUM(m.obitos) * 100.0 /
            NULLIF(SUM(m.casos),0),
            4
        ) AS taxa_obito_atual,

        CASE
            WHEN SUM(m.casos) >= 50
            THEN TRUE
            ELSE FALSE
        END AS possui_amostragem_suficiente

    FROM fiap.gold.SINAN_DENGUE_MUNICIPIOS_MENSAL m

    CROSS JOIN referencia r
    CROSS JOIN periodos p

    WHERE m.mes_ano BETWEEN
          ADD_MONTHS(r.dt_ref,-p.meses_atual)
          AND r.dt_ref

    GROUP BY
        p.periodo,
        m.cod_ibge_municipio,
        m.nome_municipio,
        r.dt_ref,
        p.meses_atual

),

anterior AS (

    SELECT

        p.periodo,

        m.cod_ibge_municipio,

        SUM(m.casos) AS casos_anterior,
        SUM(m.obitos) AS obitos_anterior,

        ROUND(
            SUM(m.obitos) * 100.0 /
            NULLIF(SUM(m.casos),0),
            4
        ) AS taxa_obito_anterior

    FROM fiap.gold.SINAN_DENGUE_MUNICIPIOS_MENSAL m

    CROSS JOIN referencia r
    CROSS JOIN periodos p

    WHERE p.possui_comparacao = TRUE

      AND m.mes_ano BETWEEN
          ADD_MONTHS(r.dt_ref,-p.meses_anterior_inicio)
          AND ADD_MONTHS(r.dt_ref,-p.meses_anterior_fim)

    GROUP BY
        p.periodo,
        m.cod_ibge_municipio

)

SELECT

    'Dengue' AS id_agravo,

    a.periodo,

    a.cod_ibge_municipio,
    a.nome_municipio,

    a.periodo_inicio,
    a.periodo_fim,

    a.casos_atual,
    a.obitos_atual,

    a.taxa_obito_atual,

    CASE
        WHEN p.possui_comparacao
        THEN b.casos_anterior
        ELSE NULL
    END AS casos_anterior,

    CASE
        WHEN p.possui_comparacao
        THEN b.obitos_anterior
        ELSE NULL
    END AS obitos_anterior,

    CASE
        WHEN p.possui_comparacao
        THEN b.taxa_obito_anterior
        ELSE NULL
    END AS taxa_obito_anterior,

    CASE
        WHEN p.possui_comparacao
             AND b.taxa_obito_anterior IS NOT NULL
        THEN ROUND(
            a.taxa_obito_atual -
            b.taxa_obito_anterior,
            4
        )
        ELSE NULL
    END AS variacao_pp,

    p.possui_comparacao AS possui_base_comparacao,

    a.possui_amostragem_suficiente,

    CASE
        WHEN a.possui_amostragem_suficiente = FALSE
        THEN 'Baixa amostragem'
        ELSE 'OK'
    END AS observacao,

    CURRENT_TIMESTAMP() AS data_referencia

FROM atual a

INNER JOIN periodos p
    ON a.periodo = p.periodo

LEFT JOIN anterior b
    ON a.periodo = b.periodo
   AND a.cod_ibge_municipio = b.cod_ibge_municipio

ORDER BY
    periodo,
    nome_municipio;



In [0]:
# Lendo a tabela do Databricks
df_taxa_obito = spark.table("fiap.silver.SINAN_DENGUE_MUNICIPIOS_TAXA_OBITO")

# Enviando para o Supabase
enviar_para_supabase(df_taxa_obito, "sinan_dengue_municipios_taxa_obito")

## CARD 04

In [0]:
%sql
CREATE OR REPLACE TABLE fiap.silver.SINAN_DENGUE_MUNICIPIOS_INCIDENCIA AS

WITH referencia AS (
SELECT MAX(mes_ano) AS dt_ref
FROM fiap.gold.SINAN_DENGUE_MUNICIPIOS_MENSAL
),

periodos AS (

SELECT
    'Trimestre' AS periodo,
    ADD_MONTHS(dt_ref,-2) AS dt_inicio_atual,
    dt_ref AS dt_fim_atual,
    ADD_MONTHS(dt_ref,-5) AS dt_inicio_anterior,
    ADD_MONTHS(dt_ref,-3) AS dt_fim_anterior,
    TRUE AS possui_base_comparacao
FROM referencia

UNION ALL

SELECT
    'Semestre',
    ADD_MONTHS(dt_ref,-5),
    dt_ref,
    ADD_MONTHS(dt_ref,-11),
    ADD_MONTHS(dt_ref,-6),
    TRUE
FROM referencia

UNION ALL

SELECT
    '12 Meses',
    ADD_MONTHS(dt_ref,-11),
    dt_ref,
    ADD_MONTHS(dt_ref,-23),
    ADD_MONTHS(dt_ref,-12),
    TRUE
FROM referencia

UNION ALL

SELECT
    '3 Anos',
    ADD_MONTHS(dt_ref,-35),
    dt_ref,
    NULL,
    NULL,
    FALSE
FROM referencia

UNION ALL

SELECT
    '5 Anos',
    ADD_MONTHS(dt_ref,-59),
    dt_ref,
    NULL,
    NULL,
    FALSE
FROM referencia

),

atual AS (


SELECT
    p.periodo,
    m.cod_ibge_municipio,
    m.nome_municipio,
    SUM(m.casos) AS casos_atual

FROM fiap.gold.SINAN_DENGUE_MUNICIPIOS_MENSAL m
CROSS JOIN periodos p

WHERE m.mes_ano BETWEEN p.dt_inicio_atual
                    AND p.dt_fim_atual

GROUP BY
    p.periodo,
    m.cod_ibge_municipio,
    m.nome_municipio


),

anterior AS (


SELECT
    p.periodo,
    m.cod_ibge_municipio,
    SUM(m.casos) AS casos_anterior

FROM fiap.gold.SINAN_DENGUE_MUNICIPIOS_MENSAL m
CROSS JOIN periodos p

WHERE p.possui_base_comparacao = TRUE
  AND m.mes_ano BETWEEN p.dt_inicio_anterior
                   AND p.dt_fim_anterior

GROUP BY
    p.periodo,
    m.cod_ibge_municipio


)

SELECT


'Dengue' AS id_agravo,

p.periodo,

a.cod_ibge_municipio,

a.nome_municipio,

p.dt_inicio_atual AS periodo_inicio,

p.dt_fim_atual AS periodo_fim,

a.casos_atual,

pop.populacao,

ROUND(
    a.casos_atual * 100000.0 /
    pop.populacao,
    2
) AS incidencia_atual,

CASE
    WHEN p.possui_base_comparacao
    THEN COALESCE(an.casos_anterior,0)
    ELSE NULL
END AS casos_anterior,

CASE
    WHEN p.possui_base_comparacao
    THEN ROUND(
        COALESCE(an.casos_anterior,0) * 100000.0 /
        pop.populacao,
        2
    )
    ELSE NULL
END AS incidencia_anterior,

CASE
    WHEN p.possui_base_comparacao
    THEN ROUND(
        (
            a.casos_atual * 100000.0 /
            pop.populacao
        )
        -
        (
            COALESCE(an.casos_anterior,0) * 100000.0 /
            pop.populacao
        ),
        2
    )
    ELSE NULL
END AS variacao_pp,

p.possui_base_comparacao,

CURRENT_TIMESTAMP() AS data_referencia


FROM atual a

INNER JOIN periodos p
ON a.periodo = p.periodo

LEFT JOIN anterior an
ON a.periodo = an.periodo
AND a.cod_ibge_municipio = an.cod_ibge_municipio

LEFT JOIN fiap.silver.IBGE_POPULACAO_SP pop
ON a.cod_ibge_municipio = pop.cod_ibge_municipio

ORDER BY
periodo,
incidencia_atual DESC;


In [0]:
# Lendo a tabela consolidada do Databricks
df_incidencia_consolidada = spark.table("fiap.silver.SINAN_DENGUE_MUNICIPIOS_INCIDENCIA")

# Enviando para o Supabase
enviar_para_supabase(df_incidencia_consolidada, "sinan_dengue_municipios_incidencia")

# Sazonalidade - Casos Mensais

In [0]:
%sql
CREATE OR REPLACE TABLE fiap.silver.SINAN_DENGUE_MUNICIPIOS_SAZONALIDADE AS

WITH referencia AS (
SELECT MAX(mes_ano) AS dt_ref
FROM fiap.gold.SINAN_DENGUE_MUNICIPIOS_MENSAL
),

periodos AS (

SELECT
    'Trimestre' AS periodo,
    ADD_MONTHS(dt_ref,-2) AS dt_inicio,
    dt_ref AS dt_fim,
    TRUE AS possui_comparacao
FROM referencia

UNION ALL

SELECT
    'Semestre',
    ADD_MONTHS(dt_ref,-5),
    dt_ref,
    TRUE
FROM referencia

UNION ALL

SELECT
    '12 Meses',
    ADD_MONTHS(dt_ref,-11),
    dt_ref,
    TRUE
FROM referencia

UNION ALL

SELECT
    '3 Anos',
    ADD_MONTHS(dt_ref,-35),
    dt_ref,
    FALSE
FROM referencia

UNION ALL

SELECT
    '5 Anos',
    ADD_MONTHS(dt_ref,-59),
    dt_ref,
    FALSE
FROM referencia


),

atual AS (


SELECT
    p.periodo,
    m.cod_ibge_municipio,
    m.nome_municipio,
    m.mes_ano,
    SUM(m.casos) AS casos_atual

FROM fiap.gold.SINAN_DENGUE_MUNICIPIOS_MENSAL m
CROSS JOIN periodos p

WHERE m.mes_ano BETWEEN p.dt_inicio
                    AND p.dt_fim

GROUP BY
    p.periodo,
    m.cod_ibge_municipio,
    m.nome_municipio,
    m.mes_ano


),

ano_anterior AS (

SELECT
    p.periodo,
    m.cod_ibge_municipio,
    ADD_MONTHS(m.mes_ano,12) AS mes_ano,
    SUM(m.casos) AS casos_ano_anterior

FROM fiap.gold.SINAN_DENGUE_MUNICIPIOS_MENSAL m
CROSS JOIN periodos p

WHERE p.possui_comparacao = TRUE
  AND m.mes_ano BETWEEN
      ADD_MONTHS(p.dt_inicio,-12)
      AND ADD_MONTHS(p.dt_fim,-12)

GROUP BY
    p.periodo,
    m.cod_ibge_municipio,
    ADD_MONTHS(m.mes_ano,12)


),

media_historica AS (

SELECT
    cod_ibge_municipio,
    MONTH(mes_ano) AS mes_num,
    AVG(casos) AS media_historica

FROM fiap.gold.SINAN_DENGUE_MUNICIPIOS_MENSAL

GROUP BY
    cod_ibge_municipio,
    MONTH(mes_ano)

)

SELECT

'Dengue' AS id_agravo,

a.periodo,

a.cod_ibge_municipio,

a.nome_municipio,

a.mes_ano,

a.casos_atual,

CASE
    WHEN a.periodo IN ('Trimestre','Semestre','12 Meses')
    THEN aa.casos_ano_anterior
    ELSE NULL
END AS casos_ano_anterior,

CASE
    WHEN a.periodo IN ('Trimestre','Semestre','12 Meses')
    THEN ROUND(mh.media_historica,2)
    ELSE NULL
END AS media_historica,

CURRENT_TIMESTAMP() AS data_referencia


FROM atual a

LEFT JOIN ano_anterior aa
ON a.periodo = aa.periodo
AND a.cod_ibge_municipio = aa.cod_ibge_municipio
AND a.mes_ano = aa.mes_ano

LEFT JOIN media_historica mh
ON a.cod_ibge_municipio = mh.cod_ibge_municipio
AND MONTH(a.mes_ano) = mh.mes_num

ORDER BY
periodo,
nome_municipio,
mes_ano;


In [0]:
# Lendo a tabela consolidada do Databricks
df_sazonalidade_consolidada = spark.table("fiap.silver.SINAN_DENGUE_MUNICIPIOS_SAZONALIDADE")

# Enviando para o Supabase
enviar_para_supabase(df_sazonalidade_consolidada, "sinan_dengue_municipios_sazonalidade")

## Distribuição por Cidade

In [0]:
%sql
CREATE OR REPLACE TABLE fiap.silver.SINAN_DENGUE_MUNICIPIOS_DISTRIBUICAO_CIDADE AS

WITH referencia AS (
SELECT MAX(mes_ano) AS dt_ref
FROM fiap.gold.SINAN_DENGUE_MUNICIPIOS_MENSAL
),

periodos AS (


SELECT
    'Trimestre' AS periodo,
    ADD_MONTHS(dt_ref,-2) AS dt_inicio,
    dt_ref AS dt_fim
FROM referencia

UNION ALL

SELECT
    'Semestre',
    ADD_MONTHS(dt_ref,-5),
    dt_ref
FROM referencia

UNION ALL

SELECT
    '12 Meses',
    ADD_MONTHS(dt_ref,-11),
    dt_ref
FROM referencia

UNION ALL

SELECT
    '3 Anos',
    ADD_MONTHS(dt_ref,-35),
    dt_ref
FROM referencia

UNION ALL

SELECT
    '5 Anos',
    ADD_MONTHS(dt_ref,-59),
    dt_ref
FROM referencia


),

base AS (


SELECT
    p.periodo,
    m.nome_municipio,
    SUM(m.casos) AS casos

FROM fiap.gold.SINAN_DENGUE_MUNICIPIOS_MENSAL m
CROSS JOIN periodos p

WHERE m.mes_ano BETWEEN p.dt_inicio
                    AND p.dt_fim

GROUP BY
    p.periodo,
    m.nome_municipio


),

ranking AS (


SELECT
    *,
    ROW_NUMBER() OVER (
        PARTITION BY periodo
        ORDER BY casos DESC
    ) AS ranking

FROM base


),

top5 AS (


SELECT
    periodo,
    nome_municipio,
    casos,
    ranking

FROM ranking

WHERE ranking <= 5


),

outros AS (


SELECT
    periodo,
    'Outros' AS nome_municipio,
    SUM(casos) AS casos,
    6 AS ranking

FROM ranking

WHERE ranking > 5

GROUP BY periodo


),

resultado AS (


SELECT * FROM top5

UNION ALL

SELECT * FROM outros


),

total AS (


SELECT
    periodo,
    SUM(casos) AS total_casos

FROM resultado

GROUP BY periodo


)

SELECT


'Dengue' AS id_agravo,

r.periodo,

r.nome_municipio,

r.casos,

ROUND(
    (r.casos / t.total_casos) * 100,
    2
) AS percentual,

r.ranking,

CURRENT_TIMESTAMP() AS data_referencia

FROM resultado r

INNER JOIN total t
ON r.periodo = t.periodo

ORDER BY
periodo,
ranking;


In [0]:
# Lendo a tabela consolidada do Databricks
df_distribuicao_cidade = spark.table("fiap.silver.SINAN_DENGUE_MUNICIPIOS_DISTRIBUICAO_CIDADE")

# Enviando para o Supabase
enviar_para_supabase(df_distribuicao_cidade, "sinan_dengue_municipios_distribuicao_cidade")

## Distribuição por gênero

In [0]:
%sql
CREATE OR REPLACE TABLE fiap.silver.sinan_dengue_municipios_distribuicao_genero AS

WITH referencia AS (

    SELECT
        MAX(mes_ano_sinan) AS dt_ref
    FROM fiap.silver.SINAN_DENGUE_SP

),

periodos AS (

    SELECT 'Trimestre' AS periodo,
           ADD_MONTHS(dt_ref,-2) AS dt_inicio,
           dt_ref AS dt_fim
    FROM referencia

    UNION ALL

    SELECT 'Semestre',
           ADD_MONTHS(dt_ref,-5),
           dt_ref
    FROM referencia

    UNION ALL

    SELECT '12 Meses',
           ADD_MONTHS(dt_ref,-11),
           dt_ref
    FROM referencia

    UNION ALL

    SELECT '3 Anos',
           ADD_MONTHS(dt_ref,-35),
           dt_ref
    FROM referencia

    UNION ALL

    SELECT '5 Anos',
           ADD_MONTHS(dt_ref,-59),
           dt_ref
    FROM referencia

),

municipios_sp AS (

    SELECT DISTINCT
        cod_ibge_municipio,
        nome_municipio
    FROM fiap.gold.SINAN_DENGUE_MUNICIPIOS_MENSAL

),

base AS (

    SELECT

        s.id_agravo,
        p.periodo,
        s.cod_ibge_municipio,
        s.genero,

        COUNT(*) AS casos

    FROM fiap.silver.SINAN_DENGUE_SP s

    INNER JOIN periodos p
        ON s.mes_ano_sinan BETWEEN p.dt_inicio AND p.dt_fim

    WHERE s.genero IN ('Feminino','Masculino')

    GROUP BY

        s.id_agravo,
        p.periodo,
        s.cod_ibge_municipio,
        s.genero

),

totais AS (

    SELECT

        id_agravo,
        periodo,
        cod_ibge_municipio,

        SUM(casos) AS total_casos

    FROM base

    GROUP BY

        id_agravo,
        periodo,
        cod_ibge_municipio

)

SELECT

    b.id_agravo,

    b.periodo,

    b.cod_ibge_municipio,

    m.nome_municipio,

    b.genero,

    b.casos,

    ROUND(
        (b.casos / t.total_casos) * 100,
        2
    ) AS percentual,

    CURRENT_DATE() AS data_referencia

FROM base b

INNER JOIN totais t

    ON b.id_agravo = t.id_agravo
   AND b.periodo = t.periodo
   AND b.cod_ibge_municipio = t.cod_ibge_municipio

INNER JOIN municipios_sp m

    ON b.cod_ibge_municipio = m.cod_ibge_municipio
;

In [0]:
# Lendo a tabela do Databricks
df_genero = spark.table("fiap.silver.SINAN_DENGUE_MUNICIPIOS_DISTRIBUICAO_GENERO")

# Enviando para o Supabase
enviar_para_supabase(df_genero, "sinan_dengue_municipios_distribuicao_genero")

## Desfecho Clínico por Ano

In [0]:
%sql
CREATE OR REPLACE TABLE fiap.silver.SINAN_DENGUE_MUNICIPIOS_DESFECHO_CLINICO_ANUAL AS

WITH municipios_sp AS (

    SELECT DISTINCT
        cod_ibge_municipio,
        nome_municipio
    FROM fiap.gold.SINAN_DENGUE_MUNICIPIOS_MENSAL

)

SELECT

    s.id_agravo,

    s.cod_ibge_municipio,

    m.nome_municipio,

    s.ano_referencia,

    SUM(
        CASE
            WHEN COALESCE(s.flag_obito_dengue,0) = 0
             AND COALESCE(s.flag_obito_geral,0) = 0
            THEN 1
            ELSE 0
        END
    ) AS casos_leves,

    SUM(
        CASE
            WHEN COALESCE(s.flag_hospitalizacao,0) = 1
            THEN 1
            ELSE 0
        END
    ) AS hospitalizacoes,

    SUM(
        CASE
            WHEN COALESCE(s.flag_obito_dengue,0) = 1
              OR COALESCE(s.flag_obito_geral,0) = 1
            THEN 1
            ELSE 0
        END
    ) AS obitos,

    CURRENT_DATE() AS data_referencia

FROM fiap.silver.SINAN_DENGUE_SP s

INNER JOIN municipios_sp m
    ON s.cod_ibge_municipio = m.cod_ibge_municipio

GROUP BY

    s.id_agravo,
    s.cod_ibge_municipio,
    m.nome_municipio,
    s.ano_referencia
;

In [0]:
# Lendo a tabela do Databricks
df_desfecho_anual = spark.table("fiap.silver.SINAN_DENGUE_MUNICIPIOS_DESFECHO_CLINICO_ANUAL")

# Enviando para o Supabase
enviar_para_supabase(df_desfecho_anual, "sinan_dengue_municipios_desfecho_clinico_anual")

## Gráfico faixa etária

In [0]:
%sql
CREATE OR REPLACE TABLE fiap.silver.SINAN_DENGUE_MUNICIPIOS_FAIXA_ETARIA AS

WITH referencia AS (
SELECT MAX(mes_ano) AS dt_ref
FROM fiap.gold.SINAN_DENGUE_MUNICIPIOS_MENSAL
),

periodos AS (


SELECT
    'Trimestre' AS periodo,
    ADD_MONTHS(dt_ref,-2) AS dt_inicio,
    dt_ref AS dt_fim
FROM referencia

UNION ALL

SELECT
    'Semestre',
    ADD_MONTHS(dt_ref,-5),
    dt_ref
FROM referencia

UNION ALL

SELECT
    '12 Meses',
    ADD_MONTHS(dt_ref,-11),
    dt_ref
FROM referencia

UNION ALL

SELECT
    '3 Anos',
    ADD_MONTHS(dt_ref,-35),
    dt_ref
FROM referencia

UNION ALL

SELECT
    '5 Anos',
    ADD_MONTHS(dt_ref,-59),
    dt_ref
FROM referencia


),

base AS (


SELECT

    p.periodo,

    s.cod_ibge_municipio,

    m.nome_municipio,

    CASE

        WHEN s.idade_anos BETWEEN 0 AND 4
            THEN '0-4'

        WHEN s.idade_anos BETWEEN 5 AND 14
            THEN '5-14'

        WHEN s.idade_anos BETWEEN 15 AND 29
            THEN '15-29'

        WHEN s.idade_anos BETWEEN 30 AND 44
            THEN '30-44'

        WHEN s.idade_anos BETWEEN 45 AND 59
            THEN '45-59'

        ELSE '60+'

    END AS faixa_etaria,

    CASE

        WHEN s.idade_anos BETWEEN 0 AND 4
            THEN 1

        WHEN s.idade_anos BETWEEN 5 AND 14
            THEN 2

        WHEN s.idade_anos BETWEEN 15 AND 29
            THEN 3

        WHEN s.idade_anos BETWEEN 30 AND 44
            THEN 4

        WHEN s.idade_anos BETWEEN 45 AND 59
            THEN 5

        ELSE 6

    END AS ordem_faixa

FROM fiap.silver.SINAN_DENGUE_SP s

INNER JOIN (
    SELECT DISTINCT
        cod_ibge_municipio,
        nome_municipio
    FROM fiap.gold.SINAN_DENGUE_MUNICIPIOS_MENSAL
) m
    ON s.cod_ibge_municipio = m.cod_ibge_municipio

CROSS JOIN periodos p

WHERE s.data_notif_date BETWEEN p.dt_inicio
                            AND p.dt_fim


),

casos_faixa AS (


SELECT

    periodo,
    cod_ibge_municipio,
    nome_municipio,
    faixa_etaria,
    ordem_faixa,

    COUNT(*) AS casos

FROM base

GROUP BY
    periodo,
    cod_ibge_municipio,
    nome_municipio,
    faixa_etaria,
    ordem_faixa


),

totais AS (


SELECT

    periodo,
    cod_ibge_municipio,

    SUM(casos) AS total_casos

FROM casos_faixa

GROUP BY
    periodo,
    cod_ibge_municipio


)

SELECT


'Dengue' AS id_agravo,

c.periodo,

c.cod_ibge_municipio,

c.nome_municipio,

c.faixa_etaria,

c.casos,

ROUND(
    (c.casos / t.total_casos) * 100,
    2
) AS percentual,

c.ordem_faixa,

CURRENT_TIMESTAMP() AS data_referencia


FROM casos_faixa c

INNER JOIN totais t
ON c.periodo = t.periodo
AND c.cod_ibge_municipio = t.cod_ibge_municipio

ORDER BY
periodo,
nome_municipio,
ordem_faixa;


In [0]:
# Lendo a tabela consolidada do Databricks
df_faixa_etaria = spark.table("fiap.silver.SINAN_DENGUE_MUNICIPIOS_FAIXA_ETARIA")

# Enviando para o Supabase
enviar_para_supabase(df_faixa_etaria, "sinan_dengue_municipios_faixa_etaria")